# 04 Public Comps and Market-Implied Debate

Oracle comps are a split debate: hyperscalers frame capacity scale,
software names frame margin durability, and AI infrastructure names frame
scarcity. An average multiple would hide the actual underwriting question.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from valuation import (
    DcfInputs,
    MarketPremiumInputs,
    OptionScenario,
    SegmentValuation,
    SotpInputs,
    build_sensitivity_grid,
    discounted_cash_flow,
    implied_margin_for_enterprise_value,
    implied_revenue_for_enterprise_value,
    market_premium_value,
    probability_weighted_option_value,
    sotp_equity_value,
)

DATA_DIR = next(
    candidate / "apps/notebooks/studies/oracle_valuation/data"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "apps/notebooks/studies/oracle_valuation/data").exists()
)
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
comps = pd.read_csv(DATA_DIR / "public_comps.csv")
comps

In [ ]:
try:
    import yfinance as yf

    rows = []
    for ticker in comps["ticker"].dropna().unique():
        info = yf.Ticker(ticker).fast_info
        rows.append(
            {
                "ticker": ticker,
                "market_cap": getattr(info, "market_cap", np.nan),
                "last_price": getattr(info, "last_price", np.nan),
                "currency": getattr(info, "currency", None),
            }
        )
    live_market = pd.DataFrame(rows)
except Exception as exc:
    live_market = pd.DataFrame(
        [
            {
                "ticker": "DATA_UNAVAILABLE",
                "market_cap": np.nan,
                "last_price": np.nan,
                "currency": str(exc),
            }
        ]
    )
live_market.head()

In [ ]:
assumptions = pd.read_csv(DATA_DIR / "segment_assumptions.csv")
market = (
    assumptions[assumptions["segment"].eq("Market")]
    .pivot_table(index="case", columns="metric", values="value", aggfunc="first")
)
current_price = market.loc["base", "current price"]
eps_guide = market.loc["base", "FY2027 adjusted EPS guide"]
current_pe = current_price / eps_guide
pd.DataFrame(
    [
        {
            "current_price": current_price,
            "FY2027_adjusted_EPS_guide": eps_guide,
            "implied_forward_PE": current_pe,
        }
    ]
)

In [ ]:
premiums = pd.read_csv(DATA_DIR / "market_premiums.csv").set_index("case")
base_fundamental_equity_value = 700.0
premium_rows = []
for case, row in premiums.iterrows():
    result = market_premium_value(
        base_fundamental_equity_value,
        MarketPremiumInputs(
            ai_scarcity_premium=row.ai_scarcity_premium,
            ipo_scarcity_premium=row.mega_cap_liquidity_premium,
            strategic_asset_premium=row.strategic_asset_premium,
            governance_discount=row.governance_discount,
            execution_haircut=row.execution_haircut,
        ),
    )
    premium_rows.append(
        {
            "case": case,
            "fundamental_equity_value_usd_bn": base_fundamental_equity_value,
            "net_premium_discount": result.net_premium,
            "market_adjusted_equity_value_usd_bn": result.market_value,
        }
    )
pd.DataFrame(premium_rows)

In [ ]:
comps.groupby("peer_group").agg(
    company_count=("company", "count"),
    avg_narrative_score=("narrative_premium_score", "mean"),
).sort_values("avg_narrative_score", ascending=False)